# Benchmark Karsilastirmasi
Portfolyo olmadan: Altin, Gumus, Dolar, Euro, BIST100 ve Mevduat faizi karsilastirmasi.
Tum varliklar baslangic=100 bazinda normalize edilir.

In [1]:
# Hucre 1 - Kurulum (Colab otomatik clone + yerel paket kontrol)
import subprocess, sys, os

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    repo_dir = "/content/BenchmarkTakip"
    if not os.path.isdir(os.path.join(repo_dir, "lib")):
        subprocess.run([
            "git", "clone",
            "https://github.com/Yusufygc/PortfoyVarliklarini_Veya_YatirimAraclari_Getiri_Kiyaslama_Benchmark_Uygulamasi.git",
            repo_dir
        ], check=True)
    os.chdir(repo_dir)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
    print(f"Colab kurulumu tamam | CWD: {os.getcwd()}")
else:
    REQUIRED = ["yfinance", "plotly", "ipywidgets"]
    for pkg in REQUIRED:
        try:
            __import__(pkg)
        except ImportError:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
    print("Bagimliliklar hazir")


Bagimliliklar hazir


In [2]:
# Hucre 2 - Importlar + Google Drive baglama
import os, sys
from datetime import datetime
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
import ipywidgets as widgets
from IPython.display import display

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive, output
    drive.mount('/content/drive')
    output.enable_custom_widget_manager()
    pio.renderers.default = "colab"
    print("Google Drive baglandi | Colab widget/Plotly renderer hazir")
else:
    print("Yerel ortam - Drive mount atlandi")

Yerel ortam - Drive mount atlandi


In [3]:
# Hucre 3 - Konfigurasyon
PROJECT_ROOT_CANDIDATES = [
    os.getcwd(),
    "/content/BenchmarkTakip",
    "/content/drive/MyDrive/PortfolioProject",
]
PROJECT_ROOT = next(
    (p for p in PROJECT_ROOT_CANDIDATES if os.path.isdir(os.path.join(p, "lib"))),
    os.getcwd(),
)

DRIVE_DATA_DIR = "/content/drive/MyDrive/PortfolioProject"
DRIVE_BASE = os.environ.get("PORTFOLIO_DATA_DIR")
if not DRIVE_BASE:
    if IN_COLAB and os.path.isdir(DRIVE_DATA_DIR):
        DRIVE_BASE = DRIVE_DATA_DIR
    else:
        DRIVE_BASE = os.path.join(PROJECT_ROOT, "data")
DRIVE_BASE = os.path.abspath(DRIVE_BASE) + os.sep

CACHE_PATH = os.path.join(DRIVE_BASE, "cache")

# Karsilastirilacak benchmark'lar
SYMBOLS = {
    "Gram Altin": "GC=F",
    "Gram Gumus": "SI=F",
    "DOLAR":      "USDTRY=X",
    "EURO":       "EURTRY=X",
    "BIST100":    "XU100.IS",
}

TCMB_POLICY_RATE_PCT = 37  # Guncelle: mevcut TCMB politika faizi

# Varsayilan secim araligi ve fiyat verisi kapsami
DATA_START = "2015-01-01"
DEFAULT_START = "2023-01-01"
DEFAULT_END   = datetime.today().strftime("%Y-%m-%d")

print(f"Config: {len(SYMBOLS)} benchmark sembol | PROJECT_ROOT={PROJECT_ROOT} | DATA={DRIVE_BASE}")

Config: 5 benchmark sembol | PROJECT_ROOT=d:\1KodCalismalari\Projeler\VIBE_CODING_UYGULAMA_DENEMELERI\BenchmarkTakip | DATA=d:\1KodCalismalari\Projeler\VIBE_CODING_UYGULAMA_DENEMELERI\BenchmarkTakip\data\


In [4]:
# Hucre 4 - lib/ import
LIB_PATH = os.path.join(PROJECT_ROOT, "lib")
if not os.path.isdir(LIB_PATH):
    raise FileNotFoundError(
        f"lib klasoru bulunamadi: {LIB_PATH}. Colab'da once repoyu clone edip os.chdir(repo_klasoru) yapin."
    )

# Notebook kernel'i ayni oturumda eski lib modullerini cache'leyebilir.
# Bu hucre her calistiginda proje lib klasorunu one alir ve modulleri yeniden yukler.
if LIB_PATH in sys.path:
    sys.path.remove(LIB_PATH)
sys.path.insert(0, LIB_PATH)

import importlib
import data_loader as _data_loader
import benchmark_engine as _benchmark_engine
import chart_builder as _chart_builder
import widgets as _widgets_module
import chart_builder_v2 as _chart_builder_v2

_data_loader = importlib.reload(_data_loader)
_benchmark_engine = importlib.reload(_benchmark_engine)
_chart_builder = importlib.reload(_chart_builder)
_widgets_module = importlib.reload(_widgets_module)
_chart_builder_v2 = importlib.reload(_chart_builder_v2)

fetch_prices = _data_loader.fetch_prices
ensure_cpi_coverage = _data_loader.ensure_cpi_coverage
load_cpi_series = _data_loader.load_cpi_series
load_tcmb_rates = _data_loader.load_tcmb_rates
load_deposit_rates = _data_loader.load_deposit_rates
get_latest_policy_rate = _data_loader.get_latest_policy_rate

build_benchmark_series = _benchmark_engine.build_benchmark_series
build_deposit_series = _benchmark_engine.build_deposit_series

build_performance_line_chart = _chart_builder.build_performance_line_chart

create_asset_selector = _widgets_module.create_asset_selector
create_date_range_picker = _widgets_module.create_date_range_picker
create_currency_toggle = _widgets_module.create_currency_toggle
wire_dashboard = _widgets_module.wire_dashboard

build_performance_line_chart_v2 = _chart_builder_v2.build_performance_line_chart_v2
build_asset_filter_widget = _chart_builder_v2.build_asset_filter_widget
build_drawdown_chart = _chart_builder_v2.build_drawdown_chart
build_correlation_heatmap = _chart_builder_v2.build_correlation_heatmap
build_period_bar_chart = _chart_builder_v2.build_period_bar_chart
build_treemap = _chart_builder_v2.build_treemap
build_risk_return_scatter = _chart_builder_v2.build_risk_return_scatter

print("lib/ moduller yeniden yuklendi")

lib/ moduller yeniden yuklendi


In [5]:
# Hucre 5 - Veri yukle
os.makedirs(CACHE_PATH, exist_ok=True)

CPI_PATH  = os.path.join(DRIVE_BASE, "cpi_turkey.csv")
TCMB_PATH    = os.path.join(DRIVE_BASE, "tcmb_rates.csv")
DEPOSIT_PATH = os.path.join(DRIVE_BASE, "deposit_rates.csv")

# tcmb_rates.csv son satirindan dinamik politika faizi (TCMB rate degisikliklerine
# notebook'u manuel guncellemeden ayak uydurur). Dosya yoksa Cell-3 fallback'i.
TCMB_POLICY_RATE_PCT = get_latest_policy_rate(TCMB_PATH, fallback=TCMB_POLICY_RATE_PCT)

cpi_update_error = None
try:
    cpi_series = ensure_cpi_coverage(CPI_PATH, DATA_START, DEFAULT_END)
except ValueError as exc:
    cpi_update_error = str(exc)
    cpi_series = load_cpi_series(CPI_PATH)

tcmb_rates    = load_tcmb_rates(TCMB_PATH, policy_rate_pct=TCMB_POLICY_RATE_PCT)
deposit_rates = load_deposit_rates(DEPOSIT_PATH, fallback_policy_rate_series=tcmb_rates)

all_symbols = list(dict.fromkeys(list(SYMBOLS.values()) + ["USDTRY=X"]))
data_load_warning = None
try:
    prices = fetch_prices(all_symbols, start=DATA_START, end=DEFAULT_END, cache_path=CACHE_PATH)
except ValueError as exc:
    data_load_warning = (
        f"Uyari: {exc} Genis tarih araligi indirilemedi; mevcut cache kapsami kullaniliyor."
    )
    prices = fetch_prices(
        all_symbols,
        start=DEFAULT_START,
        end=DEFAULT_END,
        cache_path=CACHE_PATH,
        require_start_coverage=False,
    )

fx_usdtry = prices["USDTRY=X"].dropna()

first_valid_dates = [prices[s].first_valid_index() for s in SYMBOLS.values() if s in prices.columns]
first_valid_dates = [d for d in first_valid_dates if d is not None]
if not first_valid_dates:
    raise ValueError("Fiyat verisi yuklenemedi: benchmark sembolleri icin gecerli veri yok.")

EFFECTIVE_DATA_START = max(first_valid_dates).strftime("%Y-%m-%d")
EFFECTIVE_DATA_END = prices.dropna(how="all").index.max().strftime("%Y-%m-%d")
DEFAULT_START_EFFECTIVE = max(
    pd.Timestamp(DEFAULT_START),
    pd.Timestamp(EFFECTIVE_DATA_START),
).strftime("%Y-%m-%d")
cpi_monthly_raw = pd.read_csv(CPI_PATH)
cpi_monthly_raw["Tarih"] = pd.to_datetime(cpi_monthly_raw["Tarih"], dayfirst=True)
CPI_DATA_START = cpi_monthly_raw["Tarih"].min().strftime("%Y-%m-%d")
CPI_DATA_END = cpi_monthly_raw["Tarih"].max().strftime("%Y-%m-%d")

if data_load_warning:
    print(data_load_warning)
if cpi_update_error:
    print(f"Uyari: TUFE verisi tam guncellenemedi. {cpi_update_error}")
print(
    f"Veri hazir | TCMB politika faizi: {TCMB_POLICY_RATE_PCT}% | "
    f"Gercek fiyat kapsami: {EFFECTIVE_DATA_START} - {EFFECTIVE_DATA_END} | "
    f"TUFE kapsami: {CPI_DATA_START} - {CPI_DATA_END} | "
    f"Varsayilan secim: {DEFAULT_START_EFFECTIVE} - {DEFAULT_END}"
)

Veri hazir | Gercek fiyat kapsami: 2015-01-02 - 2026-05-14 | TUFE kapsami: 2015-01-01 - 2026-04-01 | Varsayilan secim: 2023-01-01 - 2026-05-16


In [6]:
# Hucre 6 - Dashboard
output = widgets.Output()
info_panel = widgets.HTML(value="")

CURRENCY_LABELS = {"TL": "TL (Nominal)", "USD": "USD", "REAL": "Reel (TUFE)"}
MODE_DETAILS = {
    "TL": {
        "label": "TL (Nominal)",
        "basis": "Nominal TL bazinda karsilastirma",
        "formula": "Altin/Gumus = USD/troy oz x USDTRY / 31.1035; dovizler TL kuru; BIST100 puan; Mevduat nominal TL endeksi.",
        "comment": "Enflasyon etkisi dusulmez; TL cinsinden nominal buyume izlenir.",
    },
    "USD": {
        "label": "USD",
        "basis": "ABD dolari bazinda karsilastirma",
        "formula": "Altin/Gumus = USD/troy oz / 31.1035; TL bazli varliklar = TL deger / USDTRY; DOLAR yaklasik 100 referanstir.",
        "comment": "TL deger kaybi etkisi ayrilir; varliklar dolar cinsinden alim gucuyle kiyaslanir.",
    },
    "REAL": {
        "label": "Reel (TUFE)",
        "basis": "TUFE ile enflasyondan arindirilmis TL bazinda karsilastirma",
        "formula": "Once nominal TL degeri hesaplanir, sonra TL / (CPI_t / CPI_baslangic) ile deflate edilir.",
        "comment": "100 uzeri reel kazanc, 100 alti enflasyon karsisinda kayip anlamina gelir.",
    },
}
sym_to_name = {v: k for k, v in SYMBOLS.items()}
ALL_ASSET_NAMES = list(SYMBOLS.keys()) + ["Mevduat"]
active_start_date = DEFAULT_START_EFFECTIVE
active_end_date = DEFAULT_END
active_currency = "TL"
active_assets = ALL_ASSET_NAMES.copy()
analysis_df = None


def _normalize_index(series, start_date):
    series = series.dropna()
    available = series.index[series.index >= pd.Timestamp(start_date)]
    if available.empty:
        raise ValueError(f"{start_date} sonrasinda veri yok.")
    base = series.loc[available[0]]
    if base == 0:
        raise ValueError("Baz deger sifir oldugu icin normalizasyon yapilamadi.")
    return series / base * 100


def _required_cpi_end_month(end_date):
    end_month = pd.Timestamp(end_date).to_period("M").to_timestamp()
    current_month = pd.Timestamp.today().to_period("M").to_timestamp()
    if end_month == current_month:
        return current_month - pd.DateOffset(months=1)
    return end_month


def _assert_cpi_coverage(start_date, end_date):
    start_month = pd.Timestamp(start_date).to_period("M").to_timestamp()
    required_end_month = _required_cpi_end_month(end_date)
    cpi_start = pd.Timestamp(CPI_DATA_START).to_period("M").to_timestamp()
    cpi_end = pd.Timestamp(CPI_DATA_END).to_period("M").to_timestamp()
    if start_month < cpi_start or required_end_month > cpi_end:
        raise ValueError(
            "TUFE verisi secilen araligi kapsamiyor. "
            f"TUFE kapsami: {CPI_DATA_START} - {CPI_DATA_END}. "
            f"Gerekli resmi aylar: {start_month.strftime('%Y-%m')} - {required_end_month.strftime('%Y-%m')}. "
            "Hucre 5'i EVDS_API_KEY ile tekrar calistirarak veya cpi_turkey.csv dosyasini resmi endeks seviyesiyle tamamlayarak guncelleyin."
        )
    if cpi_update_error:
        print(f"Not: TUFE otomatik guncelleme uyarisi: {cpi_update_error}")


def _build_deposit_for_currency(start_date, end_date, currency):
    deposit_nominal = build_deposit_series(deposit_rates, start_date, end_date)
    if currency == "TL":
        return deposit_nominal
    if currency == "USD":
        fx_aligned = fx_usdtry.reindex(deposit_nominal.index).ffill()
        return _normalize_index(deposit_nominal / fx_aligned, start_date)
    if currency == "REAL":
        _assert_cpi_coverage(start_date, end_date)
        cpi_aligned = cpi_series.reindex(deposit_nominal.index).ffill()
        cpi_base = cpi_aligned.dropna().iloc[0]
        return _normalize_index(deposit_nominal / (cpi_aligned / cpi_base), start_date)
    raise ValueError(f"Gecersiz para birimi: {currency}")


def _mode_info_html(currency, start_date, end_date, selected_assets):
    detail = MODE_DETAILS.get(currency, MODE_DETAILS["TL"])
    selected = ", ".join(selected_assets) if selected_assets else "Secim yok"
    cpi_line = ""
    if currency == "REAL":
        cpi_line = f"<br><b>TUFE verisi:</b> {CPI_DATA_START} - {CPI_DATA_END} (son yayimlanan resmi ay ileri tasinir)"
        if cpi_update_error:
            cpi_line += " (otomatik guncelleme uyarisi var)"
    return f"""
    <div style="background:#f6f8fa;border:1px solid #d0d7de;border-radius:6px;padding:10px 12px;margin:4px 0 10px 0;max-width:1180px;font-size:13px;line-height:1.45;">
      <b>Baz:</b> {detail['basis']}<br>
      <b>Formul:</b> {detail['formula']}<br>
      <b>Yorum:</b> {detail['comment']}<br>
      <b>Secim:</b> {selected} | <b>Aralik:</b> {start_date} - {end_date}{cpi_line}
    </div>
    """


def prepare_benchmark_df(start_date, end_date, currency="TL", selected_assets=None):
    if currency == "REAL":
        _assert_cpi_coverage(start_date, end_date)

    benchmark_df = build_benchmark_series(
        symbols=list(SYMBOLS.values()),
        start_date=start_date,
        end_date=end_date,
        prices=prices,
        fx_usdtry=fx_usdtry,
        cpi_series=cpi_series if currency == "REAL" else None,
        currency=currency,
    )
    deposit_series = _build_deposit_for_currency(start_date, end_date, currency)
    if benchmark_df.empty:
        benchmark_df = pd.DataFrame(index=deposit_series.index)
    benchmark_df["Mevduat"] = deposit_series.reindex(benchmark_df.index).ffill()
    benchmark_df = benchmark_df.rename(columns=sym_to_name)

    if selected_assets is not None:
        selected_assets = [asset for asset in selected_assets if asset in benchmark_df.columns]
        benchmark_df = benchmark_df[selected_assets]

    return benchmark_df.dropna(how="all")


def set_analysis_context(start_date, end_date, currency="TL", selected_assets=None):
    global active_start_date, active_end_date, active_currency, active_assets, analysis_df
    active_start_date = start_date
    active_end_date = end_date
    active_currency = currency
    active_assets = list(selected_assets) if selected_assets is not None else ALL_ASSET_NAMES.copy()
    analysis_df = prepare_benchmark_df(start_date, end_date, currency, active_assets)
    return analysis_df


def render(start_date, end_date, currency, selected_assets):
    info_panel.value = _mode_info_html(currency, start_date, end_date, selected_assets)
    try:
        benchmark_df = set_analysis_context(start_date, end_date, currency, selected_assets)
    except ValueError as exc:
        print(f"Bu aralik icin veri hazirlanamadi: {exc}")
        return
    if benchmark_df.empty:
        print(
            "Secilen tarih araligi ve varliklar icin veri bulunamadi. "
            f"Gercek fiyat kapsami: {EFFECTIVE_DATA_START} - {EFFECTIVE_DATA_END}."
        )
        return

    currency_label = MODE_DETAILS.get(currency, MODE_DETAILS["TL"])["label"]
    title_assets = "Tum Varliklar" if set(selected_assets) == set(ALL_ASSET_NAMES) else ", ".join(selected_assets)
    chart_title = f"Benchmark Karsilastirmasi - {currency_label} - {title_assets}"

    line_fig = build_performance_line_chart(
        portfolio_series=None,
        benchmark_series=benchmark_df,
        currency_label=currency_label,
        title=chart_title,
    )
    line_fig.update_layout(
        title=dict(text=chart_title, subtitle=dict(text=MODE_DETAILS.get(currency, MODE_DETAILS["TL"])["basis"]))
    )
    display(line_fig)


start_picker, end_picker = create_date_range_picker(
    min_date=datetime.strptime(EFFECTIVE_DATA_START, "%Y-%m-%d"),
    max_date=datetime.today(),
    default_start=datetime.strptime(DEFAULT_START_EFFECTIVE, "%Y-%m-%d"),
    default_end=datetime.today(),
)
currency_toggle = create_currency_toggle()
asset_selector = create_asset_selector(ALL_ASSET_NAMES)

dashboard = wire_dashboard(
    render_fn=render,
    output_widget=output,
    start_picker=start_picker,
    end_picker=end_picker,
    currency_toggle=currency_toggle,
    asset_selector=asset_selector,
)
dashboard.children = (dashboard.children[0], info_panel, dashboard.children[1], dashboard.children[2])

with output:
    render(DEFAULT_START_EFFECTIVE, DEFAULT_END, "TL", ALL_ASSET_NAMES)

display(dashboard)

In [7]:
# Hucre 7 - Donem Sonu Getiri Ozeti
# Once Hucre 6'yi calistirin; dashboard yuklenince analysis_df otomatik set edilir.
if analysis_df is None:
    print("Once Hucre 6'yi calistirin (dashboard yuklensin).")
else:
    son_degerler = analysis_df.iloc[-1].dropna()
    getiri_df = pd.DataFrame({
        "Varlik": son_degerler.index,
        "Son Deger (baz=100)": son_degerler.values.round(2),
        "Toplam Getiri %": (son_degerler.values - 100).round(2),
    }).sort_values("Toplam Getiri %", ascending=False)

    tbl = go.Figure(go.Table(
        header=dict(
            values=["<b>Varlik</b>", "<b>Son Deger</b>", "<b>Getiri %</b>"],
            fill_color="#313244",
            font=dict(color="#cdd6f4", size=12),
            align="center",
        ),
        cells=dict(
            values=[
                getiri_df["Varlik"],
                getiri_df["Son Deger (baz=100)"],
                [f"{v:+.2f}%" for v in getiri_df["Toplam Getiri %"]],
            ],
            fill_color=[["#1e1e2e" if i % 2 == 0 else "#181825" for i in range(len(getiri_df))]],
            font=dict(color="#cdd6f4", size=11),
            align="center",
        ),
    ))
    tbl.update_layout(
        title="Donem Sonu Getiri Ozeti",
        template="plotly_dark",
        height=max(200, 38 * len(getiri_df) + 60),
        margin=dict(l=0, r=0, t=40, b=0),
    )
    display(tbl)

---
## Detayli Analiz Grafikleri

Asagidaki hucreler interaktif analiz grafikleri icerir. Her grafik oncesinde aciklama bulunur.
Grafikleri calistirmak icin once **Hucre 1-6** tamamlanmis olmali.
Dashboard'da tarih, para birimi veya varlik secimini degistirdikten sonra bu bolumdeki grafik hucrelerini yeniden calistirin.
Tum grafikler **Baslangic = 100** bazinda normalize edilmis veriyi kullanir.

In [8]:
# Analiz bolumu icin paylasilan veri seti
# Dashboard'da secilen son tarih araligi, para birimi ve varlik secimi kullanilir.
analysis_df = prepare_benchmark_df(
    active_start_date,
    active_end_date,
    active_currency,
    active_assets,
)

print(
    f"Analiz verisi hazir: {len(analysis_df)} gun, {list(analysis_df.columns)} | "
    f"Aralik: {active_start_date} - {active_end_date} | Para birimi: {CURRENCY_LABELS.get(active_currency, active_currency)}"
)

Analiz verisi hazir: 880 gun, ['Gram Altin', 'Gram Gumus', 'DOLAR', 'EURO', 'BIST100', 'Mevduat'] | Aralik: 2023-01-01 - 2026-05-16 | Para birimi: TL (Nominal)


### Performans Karşılaştırması (İnteraktif Filtre)

**Ne gösterir:** Tüm varlıkların başlangıç tarihinden bu yana normalize edilmiş getiri seyrini kıyaslar.
Tüm çizgiler **başlangıç = 100** baz alınarak yeniden ölçeklendirilmiştir; böylece farklı birimler
(TL/gram, dolar, puan) doğrudan karşılaştırılabilir. Kesikli çizgi kullanılmaz — tüm seriler solid.

**Nasıl kullanılır:**
- **Combobox:** Dropdown menüsünden tek bir varlık seçerek yalnızca o varlığın performansını inceleyin.
- **Range Selector:** Grafiğin üstündeki butonlarla (1A · 3A · 6A · YBB · 1Y · Tümü) zaman aralığını daraltın.
- **Range Slider:** Grafik altındaki mini çubuğu sürükleyerek özel bir zaman penceresi seçin.
- **Hover:** İmleç grafik üzerindeyken tarih, güncel değer ve başlangıçtan % değişimi okunur.
- **Legend:** Bir varlık adına tıklayarak o çizgiyi gizleyin; çift tıkla ile sadece onu görüntüleyin.

In [9]:
# Interaktif performans grafigi - coklu secim dashboard'da yapilir.
# Bu hucre, dashboard'da secilen aktif varliklarla detayli performans filtresini acar.
filter_widget = build_asset_filter_widget(
    benchmark_df=analysis_df,
    portfolio_series=None,
    currency_label=CURRENCY_LABELS.get(active_currency, active_currency),
)
display(filter_widget)

### Maksimum Drawdown — Tepe'den Düşüş

**Ne gösterir:** Her varlığın kendi tarihsel tepe noktasından ne kadar geride kaldığını yüzde olarak gösterir.
Sıfır çizgisi o varlığın bir önceki tarihsel zirvesini temsil eder; aşağı yönlü bölgeler kayıp dönemlerini
işaret eder. Tepe yeniden kırıldığı anda değer sıfıra döner.

**Neden önemlidir:**
- Yatırımcının tahammül etmesi gereken maksimum geçici kaybı (max-drawdown) görselleştirir.
- İki varlık benzer toplam getiri sağlasa bile farklı risk profillerine sahip olabilir;
  daha düz seyreden varlık genellikle daha az psikolojik baskı yaratır.
- Portföy varsa kırmızı alan dolgusuyla vurgulanır.

In [10]:
# Drawdown grafiği
# Y ekseni: tepe noktasından yüzde düşüş (0 = tepede, negatif = kayıp bölgesi)
fig_dd = build_drawdown_chart(analysis_df)
fig_dd.show()

### Varlık Getiri Korelasyonu

**Ne gösterir:** Günlük yüzde getiriler arasındaki istatistiksel ilişkiyi ölçer.
Renk skalası: **kırmızı → −1** (tam zıt hareket) · **koyu zemin → 0** (ilişkisiz) · **yeşil → +1** (aynı yön).
Köşegen her zaman **1.00** olmalıdır (varlığın kendisiyle olan korelasyonu).

**Neden önemlidir:**
- Korelasyonu düşük veya negatif varlıklar bir arada tutulduğunda portföy volatilitesi azalır.
- Yüksek korelasyonlu iki varlığı birden tutmak gerçek anlamda çeşitlendirme sağlamaz.
- Örneğin Gram Altın ile Dolar/TL arasındaki korelasyon, TL değer kaybı dönemlerini anlamak
  için önemli bir sinyal olabilir.

In [11]:
# Korelasyon ısı haritası
# Günlük getiri değişimleri kullanılarak Pearson korelasyon matrisi hesaplanır
fig_corr = build_correlation_heatmap(analysis_df)
fig_corr.show()

### Dönemsel Getiri Karşılaştırması (Aylık / Çeyreklik)

**Ne gösterir:** Her dönemin (ay veya çeyrek) son günündeki kapanış değeri üzerinden hesaplanan
yüzde getiriyi gruplanmış çubuk grafik olarak sunar. Hover'da ilgili dönemin tam getirisi okunur.

**Nasıl okunur:**
- **Sıfır çizgisinin üstü:** o dönemde kazanç.
- **Sıfır çizgisinin altı:** o dönemde değer kaybı.
- Aynı dönemde farklı renkteki çubukları kıyaslayarak hangi varlığın öne çıktığını görün.
- Örneğin yüksek enflasyon dönemlerinde Altın çubuklarının diğerlerine göre konumuna bakın.

In [12]:
# Aylık dönemsel getiri karşılaştırması
fig_monthly = build_period_bar_chart(analysis_df, freq="ME")
fig_monthly.show()

# Çeyreklik dönemsel getiri karşılaştırması
fig_quarterly = build_period_bar_chart(analysis_df, freq="QE")
fig_quarterly.show()

### Katkı Haritası ve Risk-Getiri Dağılımı

**Katkı Haritası (Treemap):**
Her kutunun **büyüklüğü** varlığın mutlak toplam getirisini (P&L), **rengi** ise yüzde getirisini gösterir.
Yeşil = kazanç · Kırmızı = kayıp. Hover'da P&L tutarı, yüzde getiri, ağırlık ve portföy katkısı görünür.

**Risk-Getiri Dağılımı (Scatter):**
Her nokta bir varlığı temsil eder. **X ekseni** yıllık volatilite (risk ölçütü),
**Y ekseni** başlangıçtan bu yana toplam getiriyi gösterir.
**Hedef konum: sol üst** (yüksek getiri + düşük risk = verimli varlık).
Sağ alt köşedeki noktalar yüksek risk taşıyıp düşük getiri sunuyor demektir.
Portföy mevcutsa yıldız (★) sembolüyle gösterilir.

In [13]:
# Katki haritasi - toplam getiri bazli (portfoy verisi yoksa sentetik hesaplanir)
if analysis_df.empty:
    raise ValueError("Katki haritasi icin analysis_df bos. Once dashboard'da en az bir varlik secin.")

_last  = analysis_df.iloc[-1]
_first = analysis_df.iloc[0]
_ret   = (_last / _first - 1) * 100
_n     = len(analysis_df.columns)
_weight = 100.0 / _n

contributions_analysis = pd.DataFrame({
    "Varlik Adi":       analysis_df.columns.tolist(),
    "pnl_tl":           (_ret * 1000).tolist(),
    "pnl_pct":          _ret.tolist(),
    "weight_pct":       [_weight] * _n,
    "contribution_pct": (_weight / 100 * _ret).tolist(),
})
contributions_analysis = contributions_analysis.rename(columns={"Varlik Adi": "Varl\u0131k Ad\u0131"})

# Treemap: buyukluk = mutlak P&L, renk = yuzde getiri
fig_treemap = build_treemap(contributions_analysis)
fig_treemap.show()

# Risk-getiri scatter: her nokta bir varlik
fig_scatter = build_risk_return_scatter(analysis_df)
fig_scatter.show()